# Silver Layer — Clean & Enrich Quality Data
Deduplicate, validate, and add derived quality metrics.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, avg, stddev, abs as spark_abs,
    to_timestamp, date_format, hour, dayofweek,
    row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Read bronze sensor data and deduplicate
df_sensors = spark.read.format('delta').table('bronze_sensor_readings')

# Deduplicate on reading_id (keep latest ingestion)
w = Window.partitionBy('reading_id').orderBy(col('ingestion_timestamp').desc())
df_sensors = (
    df_sensors
    .withColumn('_rn', row_number().over(w))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

# Cast and validate
df_sensors = (
    df_sensors
    .withColumn('reading_timestamp', to_timestamp('reading_timestamp'))
    .withColumn('temperature', col('temperature').cast('double'))
    .withColumn('pressure', col('pressure').cast('double'))
    .withColumn('vibration', col('vibration').cast('double'))
    .withColumn('humidity', col('humidity').cast('double'))
    .filter(col('reading_timestamp').isNotNull())
    .filter(col('temperature').between(-50, 500))
    .filter(col('pressure').between(0, 2000))
)

# Add derived columns
df_sensors = (
    df_sensors
    .withColumn('reading_date', date_format('reading_timestamp', 'yyyy-MM-dd'))
    .withColumn('reading_hour', hour('reading_timestamp'))
    .withColumn('shift', when(hour('reading_timestamp') < 8, 'Night')
                        .when(hour('reading_timestamp') < 16, 'Day')
                        .otherwise('Evening'))
    # Flag anomalous readings (> 2 std devs from mean per machine)
    .withColumn('temp_anomaly', lit(False))
    .withColumn('vibration_anomaly', lit(False))
    .withColumn('silver_timestamp', current_timestamp())
)

# Calculate per-machine stats for anomaly detection
stats = df_sensors.groupBy('machine_id').agg(
    avg('temperature').alias('avg_temp'),
    stddev('temperature').alias('std_temp'),
    avg('vibration').alias('avg_vib'),
    stddev('vibration').alias('std_vib')
)

df_sensors = df_sensors.join(stats, 'machine_id', 'left')
df_sensors = (
    df_sensors
    .withColumn('temp_anomaly',
                spark_abs(col('temperature') - col('avg_temp')) > 2 * col('std_temp'))
    .withColumn('vibration_anomaly',
                spark_abs(col('vibration') - col('avg_vib')) > 2 * col('std_vib'))
    .drop('avg_temp', 'std_temp', 'avg_vib', 'std_vib')
)

df_sensors.write.mode('overwrite').format('delta').saveAsTable('silver_sensor_readings')
print(f'Silver sensor readings: {spark.table("silver_sensor_readings").count()} rows')

In [ ]:
# Read and clean production batches
df_batches = spark.read.format('delta').table('bronze_production_batches')

w = Window.partitionBy('batch_id').orderBy(col('ingestion_timestamp').desc())
df_batches = (
    df_batches
    .withColumn('_rn', row_number().over(w))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

# Cast and derive
df_batches = (
    df_batches
    .withColumn('batch_start', to_timestamp('batch_start'))
    .withColumn('batch_end', to_timestamp('batch_end'))
    .withColumn('units_produced', col('units_produced').cast('int'))
    .withColumn('defect_count', col('defect_count').cast('int'))
    .withColumn('planned_units', col('planned_units').cast('int'))
    .withColumn('downtime_minutes', col('downtime_minutes').cast('double'))
    .filter(col('units_produced') >= 0)
    .filter(col('defect_count') >= 0)
    # Derive quality metrics
    .withColumn('yield_pct',
                when(col('units_produced') > 0,
                     (col('units_produced') - col('defect_count')) / col('units_produced') * 100)
                .otherwise(0))
    .withColumn('defect_rate',
                when(col('units_produced') > 0,
                     col('defect_count') / col('units_produced') * 100)
                .otherwise(0))
    .withColumn('production_date', date_format('batch_start', 'yyyy-MM-dd'))
    .withColumn('shift', when(hour('batch_start') < 8, 'Night')
                        .when(hour('batch_start') < 16, 'Day')
                        .otherwise('Evening'))
    .withColumn('silver_timestamp', current_timestamp())
)

df_batches.write.mode('overwrite').format('delta').saveAsTable('silver_production_batches')
print(f'Silver production batches: {spark.table("silver_production_batches").count()} rows')

In [ ]:
# Pass through equipment catalog (already clean dimension)
df_equipment = spark.read.format('delta').table('bronze_equipment_catalog')
df_equipment = df_equipment.withColumn('silver_timestamp', current_timestamp())
df_equipment.write.mode('overwrite').format('delta').saveAsTable('silver_equipment_catalog')
print(f'Silver equipment catalog: {spark.table("silver_equipment_catalog").count()} rows')